# Corrección de sobreajuste: versión 002

El experimento reproducible está en `src/regularizar_modelo.py`. Desde la raíz, ejecutar `.venv/bin/python -m src.regularizar_modelo` para entrenar y luego `.venv/bin/python src/notebook_regularizacion.py` para regenerar este análisis con kernel limpio.

El usuario priorizó reducir sobreajuste. Antes de ejecutar se fijó una regla: entre modelos a un punto porcentual o menos de la mejor accuracy CV, elegir la menor brecha entrenamiento-validación. Se limita vocabulario y frecuencia mínima y se compara regularización. No se usa la prueba local ya examinada para ajustar ni evaluar de nuevo. Las cifras siguientes son CV de selección, no una nueva estimación independiente.

In [1]:
from pathlib import Path
import os, json
import pandas as pd
from IPython.display import display, Markdown
if not Path('reports/registro_experimento_002.json').exists():
    os.chdir('..')
r=json.loads(Path('reports/registro_experimento_002.json').read_text())
display(Markdown(Path('reports/correccion_sobreajuste_002.md').read_text()))

# Corrección de sobreajuste

La regla se guardó antes de la búsqueda en plan_regularizacion_002.json: Entre configuraciones a <=0.01 de la máxima accuracy CV, elegir la menor brecha media entrenamiento-validación (truncada en cero); desempatar por accuracy CV descendente. Regla fijada antes de ejecutar. No usar prueba local.

Se compararon 25 configuraciones en los mismos cinco folds de desarrollo (9600 filas). TF-IDF permanece dentro del Pipeline. Se probaron vocabularios limitados, min_df=5 y distintos niveles de regularización. El modelo original participa como referencia. No se usa eval para ajustar vocabulario ni seleccionar modelos.

Configuración elegida: `{'clf__C': 4.0, 'tfidf__max_features': 1000, 'tfidf__min_df': 5, 'tfidf__ngram_range': (1, 1)}`.

| Accuracy | Original | Corrección |
|---|---:|---:|
| Entrenamiento (media folds) | 0.982526 | 0.823568 |
| Validación (media folds) | 0.731979 | 0.726042 |
| Brecha | 0.250547 | 0.097526 |

F1 macro CV corregido: 0.737525 ± 0.007455. Accuracy CV: 0.726042 ± 0.008149.

La tolerancia de un punto porcentual limita la pérdida de validación permitida para favorecer una brecha menor. No se minimiza la brecha sin restricciones: un modelo que siempre predice la mayoría puede tener una brecha pequeña y ser inútil. La mejora de la brecha se refiere a estos folds; no demuestra ausencia absoluta de sobreajuste. La selección entre múltiples configuraciones puede introducir optimismo en CV, y reutiliza los folds del experimento anterior.

No se vuelve a evaluar la prueba local original: ya se observó y motivó esta revisión. Su accuracy anterior NO corresponde al modelo corregido. La evaluación independiente del nuevo modelo queda pendiente. No se garantiza un puntaje en Kaggle y no se realizó ningún envío.

Tras cerrar parámetros, se ajustó sobre todo train para producción. El CSV nuevo tiene 3000 filas con IDs en el orden requerido y predicciones verificadas tras recargar el modelo. Los originales se conservan. Archivo recomendado para enviar manualmente: submissions/submission_002_regularizado.csv. El número identifica una versión local, no un envío efectuado.


In [2]:
cv=pd.read_csv('reports/resultados_cv_regularizacion_002.csv')
cols=['params','mean_train_accuracy','mean_test_accuracy','std_test_accuracy','mean_test_f1_macro','accuracy_gap','selected']
with pd.option_context('display.max_rows',30,'display.max_colwidth',180):
    display(cv[cols].sort_values('mean_test_accuracy',ascending=False))
display(r['checks'])
display(r['artifact_sha256'])

,params,mean_train_accuracy,mean_test_accuracy,std_test_accuracy,mean_test_f1_macro,accuracy_gap,selected
24,"{'clf__C': 4.0, 'tfidf__max_features': None, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2), 'tfidf__sublinear_tf': False}",0.982526,0.731979,0.007363,0.743487,0.250547,False
23,"{'clf__C': 4.0, 'tfidf__max_features': 10000, 'tfidf__min_df': 5, 'tfidf__ngram_range': (1, 2)}",0.954427,0.730104,0.010160,0.741825,0.224323,False
21,"{'clf__C': 4.0, 'tfidf__max_features': 3000, 'tfidf__min_df': 5, 'tfidf__ngram_range': (1, 2)}",0.874323,0.728542,0.006534,0.740534,0.145781,False
18,"{'clf__C': 4.0, 'tfidf__max_features': 1000, 'tfidf__min_df': 5, 'tfidf__ngram_range': (1, 1)}",0.823568,0.726042,0.008149,0.737525,0.097526,True
22,"{'clf__C': 4.0, 'tfidf__max_features': 10000, 'tfidf__min_df': 5, 'tfidf__ngram_range': (1, 1)}",0.857995,0.724896,0.006077,0.736455,0.133099,False
20,"{'clf__C': 4.0, 'tfidf__max_features': 3000, 'tfidf__min_df': 5, 'tfidf__ngram_range': (1, 1)}",0.857995,0.724896,0.006077,0.736455,0.133099,False
15,"{'clf__C': 1.0, 'tfidf__max_features': 3000, 'tfidf__min_df': 5, 'tfidf__ngram_range': (1, 2)}",0.837187,0.720521,0.007771,0.731998,0.116667,False
14,"{'clf__C': 1.0, 'tfidf__max_features': 3000, 'tfidf__min_df': 5, 'tfidf__ngram_range': (1, 1)}",0.824792,0.720417,0.006450,0.731149,0.104375,False
16,"{'clf__C': 1.0, 'tfidf__max_features': 10000, 'tfidf__min_df': 5, 'tfidf__ngram_range': (1, 1)}",0.824792,0.720417,0.006450,0.731149,0.104375,False
17,"{'clf__C': 1.0, 'tfidf__max_features': 10000, 'tfidf__min_df': 5, 'tfidf__ngram_range': (1, 2)}",0.893620,0.720000,0.009786,0.731186,0.173620,False


{'group_disjointness': True,
 'csv_schema_ids_order_labels_roundtrip': True,
 'reloaded_predictions_identical': True}

{'models/modelo_002_regularizado.joblib': 'b7c65f0c6fe7453eee88c8c27526af2c9dd14ee537ffd901beb16a62870e6949',
 'submissions/submission_002_regularizado.csv': '34c85f95c921a8985369ec6af6341c1812731e3b4ca130fe9dc045a3d92d81dd',
 'reports/particiones.csv': 'd90cbd30a92a366298a66f2322df4869694798392f19b9c0978dbecac104a931'}

## Interpretación y límites
Una brecha menor sugiere menos adaptación al entrenamiento; no basta por sí sola para demostrar mejor generalización. El límite de pérdida en validación evita elegir una solución muy débil solo por una brecha pequeña. No se garantiza ausencia de sobreajuste, ni independencia de reseñas que compartan plantillas o procedencia desconocida.

El modelo final se ajusta sobre todo train después de seleccionar parámetros. El antiguo resultado de prueba local pertenece al modelo anterior, nunca al modelo corregido. El CSV original se conserva para trazabilidad; la versión 002 puede ser el primer envío real si aún no se ha subido ninguno.